# Issue 3.1 — Full SASRec tuning and winner repeats

Abhinita Sanabada · Part D

Your GPU smoke test passed. This notebook runs the eight configurations on all
users with seed 42, then trains the selected configuration with seeds 43 and 44.
Maximum 200 epochs per run, with patience 10. Selection and checkpoint verification
use full-catalog validation NDCG@10. Final test evaluation remains the Part E handoff.

1. Upload this notebook to Colab and select a GPU runtime.
2. Confirm Workbook section 4.4 is committed, then set the confirmation below to True.
3. Run cells in order, including repeat training and the final verification cell.
4. Keep the entire output folder in Drive for the team's final evaluation.

Inputs use the shared Drive folder ID that passed your smoke test. No shortcut path
or GitHub upload is needed. All training code is bundled unchanged from the smoke
notebook and checked against the local project when this notebook is prepared.
Results and resume checkpoints are saved directly to Drive. Full training can take
substantially longer than smoke; no completion time has been measured.

Do not change the model, batch size, optimizer, or other fixed settings between runs.
If Colab disconnects, preserve the output folder and its last.pt files. The grid
runner does not automatically resume a partial grid: do not blindly launch it again.


## 1. Confirm the team prerequisite and full-run settings


In [ ]:
from pathlib import Path

# Team folder from your shared link; no My Drive shortcut is needed for inputs.
DRIVE_DATA_FOLDER_ID = "11-UfjS51JY1ylATnBm7wcLfDpueOL637"
DRIVE_RESULTS_DIR = Path("/content/drive/MyDrive/298A/part_d_grid_results")
PROJECT = Path("/content/issue2_grid_project")
RUN_MODE = "full"  # the separate 09 notebook remains available for smoke tests
assert RUN_MODE in ("smoke", "full")
BATCH_SIZE = 128
EPOCHS = 2 if RUN_MODE == "smoke" else 200
NUM_LAYERS = 2
NUM_HEADS = 1
WEIGHT_DECAY = 0.0
PATIENCE = 10
TUNING_SEED = 42
OPTIMIZER = "Adam"
TRAIN_USERS = 128 if RUN_MODE == "smoke" else None
VALIDATION_USERS = 128 if RUN_MODE == "smoke" else None
EVAL_BATCH_SIZE = 128
DEVICE = "auto"
DISCARD_RESUME_CHECKPOINTS = RUN_MODE == "smoke"

# Team settings agreed 2026-11-06; no LR scheduler, embedding std 0.02.
# Confirm Workbook 2 Section 4.4 is committed before launching full experiments.

METRICS_4_4_COMMITTED = False  # set True only after the team confirms it is committed


## 2. Mount Drive and prepare the runtime
The declared environment is Python 3.13.16 / PyTorch 2.14.1+cu130, matching the
recorded November 7 full run, with NumPy 2.2.6, pandas 2.3.2 and PyArrow 21.0.0.
This is a strict preflight target, not an installer for Python/PyTorch or an
assumption about today's Colab defaults. The checks below stop on any mismatch
before grid or repeat training. If this runtime is unavailable, agree on a new
explicit target and a separate experiment; do not mix environments across seeds.
The repository's Python 3.11.13 / PyTorch 2.8.0 pins remain the local reference.
Historical run records are preserved. Environment reports include actual versions,
GPU/CUDA details and pip freeze; they do not guarantee bitwise reproducibility.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import subprocess
import sys
import os

for folder in ("src/data", "src/models", "src/training", "src/evaluation", "configs", "data/processed"):
    (PROJECT / folder).mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "numpy==2.2.6", "pandas==2.3.2", "pyarrow==21.0.0"], check=True)
subprocess.run([sys.executable, "-c",
    "import torch,numpy,pandas,pyarrow; print('torch',torch.__version__); "
    "print('numpy',numpy.__version__,'pandas',pandas.__version__,'pyarrow',pyarrow.__version__); "
    "print('CUDA available:',torch.cuda.is_available())"], check=True)
os.environ["OMP_NUM_THREADS"] = "2"


## 3. Write the bundled project code
Run all cells in this section. These are readable, editable copies of the local project modules. They overwrite only their corresponding files under `/content/issue2_grid_project`. You can collapse this section after setup.

### src/data/sequence_dataset.py

In [ ]:
%%writefile /content/issue2_grid_project/src/data/sequence_dataset.py
"""Importable Part B adapter, derived from sequence_dataloader.ipynb sections 2-3.

The teammate notebook is unchanged. No Drive mount, file writes, model training,
or evaluator is executed on import. Keep sliding_window=False for the baseline.
"""
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader


def left_pad(seq, max_len):
    seq = np.asarray(seq, dtype=np.int64)[-max_len:]
    out = np.zeros(max_len, dtype=np.int64)
    if len(seq):
        out[max_len - len(seq):] = seq
    return out


def training_windows(items, max_len, sliding_window=False):
    items = list(items)
    if not sliding_window:
        return [items[-(max_len + 1):]]
    windows, end = [], len(items)
    while end >= 2:
        windows.append(items[max(0, end - max_len - 1):end])
        end -= max_len
    return windows


def build_train_arrays(sequences, max_len, sliding_window=False):
    users, inputs, targets = [], [], []
    for u, items in zip(sequences.user_idx.to_numpy(), sequences.train_items):
        for w in training_windows(items, max_len, sliding_window):
            users.append(u)
            inputs.append(left_pad(w[:-1], max_len))
            targets.append(left_pad(w[1:], max_len))
    return np.asarray(users, dtype=np.int64), np.stack(inputs), np.stack(targets)


def build_eval_arrays(sequences, split, max_len):
    if split == 'val':
        histories = [np.asarray(t, dtype=np.int64) for t in sequences.train_items]
        targets = sequences.val_item.to_numpy(dtype=np.int64)
    elif split == 'test':
        histories = [np.append(np.asarray(t, dtype=np.int64), v)
                     for t, v in zip(sequences.train_items, sequences.val_item)]
        targets = sequences.test_item.to_numpy(dtype=np.int64)
    else:
        raise ValueError("split must be 'val' or 'test'")
    inputs = np.stack([left_pad(h, max_len) for h in histories])
    return sequences.user_idx.to_numpy(dtype=np.int64), inputs, targets, histories


class SequenceDataset(Dataset):
    def __init__(self, sequences, mode='train', max_len=50, sliding_window=False):
        if mode not in ('train', 'val', 'test'):
            raise ValueError("mode must be 'train', 'val' or 'test'")
        self.mode, self.max_len = mode, max_len
        if mode == 'train':
            users, inputs, targets = build_train_arrays(sequences, max_len, sliding_window)
            self.histories = None
        else:
            users, inputs, targets, histories = build_eval_arrays(sequences, mode, max_len)
            # Parquet-backed NumPy views can be read-only; own these tensors safely.
            self.histories = [torch.from_numpy(h.copy()) for h in histories]
        self.user_idx = torch.from_numpy(users)
        self.inputs = torch.from_numpy(inputs)
        self.targets = torch.from_numpy(targets)

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, i):
        if self.mode == 'train':
            return {'user_idx': self.user_idx[i], 'input_seq': self.inputs[i],
                    'target_seq': self.targets[i]}
        return {'user_idx': self.user_idx[i], 'input_seq': self.inputs[i],
                'target': self.targets[i], 'history': self.histories[i]}


def eval_collate(batch):
    return {
        'user_idx': torch.stack([b['user_idx'] for b in batch]),
        'input_seq': torch.stack([b['input_seq'] for b in batch]),
        'target': torch.stack([b['target'] for b in batch]),
        'history': torch.nn.utils.rnn.pad_sequence(
            [b['history'] for b in batch], batch_first=True, padding_value=0),
    }


def make_dataloaders(sequences, max_len=50, batch_size=256, eval_batch_size=512,
                     num_workers=0, seed=42, sliding_window=False, pin_memory=None):
    pin_memory = torch.cuda.is_available() if pin_memory is None else pin_memory
    generator = torch.Generator().manual_seed(seed)
    return {
        'train': DataLoader(SequenceDataset(sequences, 'train', max_len, sliding_window),
                            batch_size=batch_size, shuffle=True, generator=generator,
                            drop_last=False, num_workers=num_workers, pin_memory=pin_memory),
        **{split: DataLoader(SequenceDataset(sequences, split, max_len),
                             batch_size=eval_batch_size, shuffle=False,
                             collate_fn=eval_collate, num_workers=num_workers,
                             pin_memory=pin_memory) for split in ('val', 'test')},
    }


### src/models/sasrec.py

In [ ]:
%%writefile /content/issue2_grid_project/src/models/sasrec.py
"""Configurable adaptation of the team's Drive Notebook 08 TinySASRec.

Not an independently validated reproduction of the original SASRec paper.
"""
import torch
from torch import nn


class SASRec(nn.Module):
    def __init__(self, n_items, max_len=50, hidden_size=64, num_heads=4,
                 num_layers=2, dropout=0.2):
        super().__init__()
        if hidden_size % num_heads:
            raise ValueError('num_heads must divide hidden_size')
        self.num_heads = num_heads
        self.item_embedding = nn.Embedding(n_items + 1, hidden_size, padding_idx=0)
        self.position_embedding = nn.Embedding(max_len, hidden_size)
        layer = nn.TransformerEncoderLayer(hidden_size, num_heads, 4 * hidden_size,
                    dropout=dropout, activation='gelu', batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(hidden_size)
        # Small initial embeddings: with tied output weights, N(0,1) starts the loss far above chance.
        for embedding in (self.item_embedding, self.position_embedding):
            nn.init.normal_(embedding.weight, mean=0.0, std=0.02)
        with torch.no_grad():
            self.item_embedding.weight[0].zero_()

    def hidden(self, tokens):
        batch, length = tokens.shape
        positions = torch.arange(length, device=tokens.device)
        padding = tokens.eq(0)
        h = self.item_embedding(tokens) + self.position_embedding(positions)
        # Left-padding + causal attention otherwise gives padded queries no allowed
        # keys, causing NaNs in some attention backends. Let only padded queries
        # attend themselves; real queries still cannot attend padding or future tokens.
        mask = torch.ones(length, length, dtype=torch.bool, device=tokens.device).triu(1)
        mask = mask.unsqueeze(0).expand(batch, -1, -1) | padding.unsqueeze(1)
        mask = mask & ~(padding.unsqueeze(2) & torch.eye(length, dtype=torch.bool, device=tokens.device))
        mask = mask.repeat_interleave(self.num_heads, dim=0)
        h = self.encoder(h, mask=mask)
        return self.norm(h).masked_fill(padding.unsqueeze(2), 0.0)

    def forward(self, tokens):
        return self.hidden(tokens) @ self.item_embedding.weight.T

    def next_item_scores(self, tokens):
        return self.hidden(tokens)[:, -1] @ self.item_embedding.weight.T

    def training_logits(self, tokens, targets):
        # Avoid materializing batch x length x catalog logits for padding positions.
        keep = targets.ne(0)
        return self.hidden(tokens)[keep] @ self.item_embedding.weight.T, targets[keep]


### src/training/config.py

In [ ]:
%%writefile /content/issue2_grid_project/src/training/config.py
"""Validate Part D settings and enumerate configurations without training a model."""
import argparse
import copy
import itertools
import json
import math
from pathlib import Path

GRID = {"hidden_size": [64, 128], "dropout": [0.2, 0.5],
        "learning_rate": [0.001, 0.0005]}
INTEGER_SETTINGS = ("batch_size", "epochs", "num_layers", "num_heads", "early_stopping_patience")


def validate_config(config, require_ready=False):
    """Allow explicit nulls in a planning template; reject them before execution."""
    def require(condition, message):
        if not condition:
            raise ValueError(message)
    require(isinstance(config, dict), "Configuration must be an object")
    require(type(config.get("schema_version")) is int and config["schema_version"] == 1,
            "schema_version must be 1")
    for section in ("data", "training", "search", "evaluation"):
        require(isinstance(config.get(section), dict), f"Missing object: {section}")
    require(set(config["search"]) == set(GRID), "Search must contain exactly the three assigned parameters")
    for key, expected in GRID.items():
        values = config["search"][key]
        require(isinstance(values, list) and len(values) == 2 and
                all(type(x) in (int, float) for x in values) and
                set(values) == set(expected), f"search.{key} must contain {expected} exactly once")
    data, training = config["data"], config["training"]
    require(type(data.get("max_seq_len")) is int and data["max_seq_len"] == 50,
            "data.max_seq_len must match the frozen dataset: 50")
    for key in ("expected_dataset", "expected_subset"):
        require(isinstance(data.get(key), str) and bool(data[key].strip()), f"Missing data.{key}")
    for key in INTEGER_SETTINGS:
        require(key in training, f"Missing training.{key}")
        value = training[key]
        require(value is None or (type(value) is int and value > 0), f"training.{key} must be a positive integer or null")
    require("weight_decay" in training, "Missing training.weight_decay")
    value = training["weight_decay"]
    require(value is None or (type(value) in (int, float) and math.isfinite(value) and value >= 0),
            "weight_decay must be finite and nonnegative or null")
    require("seeds" in training, "Missing training.seeds")
    seeds = training["seeds"]
    require(seeds is None or (isinstance(seeds, list) and len(seeds) > 0 and
            all(type(s) is int and 0 <= s < 2**32 for s in seeds) and len(set(seeds)) == len(seeds)),
            "seeds must be unique integers in [0, 2**32), or null")
    if 'optimizer' in training:
        require(training['optimizer'] in ('Adam', 'AdamW'), 'optimizer must be Adam or AdamW')
    if 'scheduler' in training:
        require(training['scheduler'] == 'none', 'Only scheduler=none is supported by this grid')
    heads = training["num_heads"]
    require(heads is None or all(size % heads == 0 for size in GRID["hidden_size"]),
            "num_heads must divide both 64 and 128")
    require(config["evaluation"] == {"selection_split": "val", "selection_metric": "NDCG@10", "sampled": False},
            "Selection must use full-catalog validation NDCG@10")
    require(isinstance(config.get("output_dir"), str) and bool(config["output_dir"].strip()), "output_dir is required")
    require("processed_dir" in data and (data["processed_dir"] is None or
            isinstance(data["processed_dir"], str) and bool(data["processed_dir"].strip())),
            "data.processed_dir must be a nonempty path or null")
    pending = [f"training.{k}" for k in (*INTEGER_SETTINGS, "weight_decay", "seeds") if training[k] is None]
    if data["processed_dir"] is None:
        pending.append("data.processed_dir")
    if require_ready:
        require(not pending, "Unconfirmed settings: " + ", ".join(pending))
    return pending


def load_config(path, require_ready=False):
    with Path(path).open() as stream:
        config = json.load(stream)
    validate_config(config, require_ready)
    return config


def expand_grid(config):
    validate_config(config)
    runs = []
    for values in itertools.product(*(config["search"][k] for k in GRID)):
        run = copy.deepcopy(config)
        run.pop("search")
        run["model_hyperparameters"] = dict(zip(GRID, values))
        runs.append(run)
    return runs


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", default="configs/config.json")
    parser.add_argument("--require-ready", action="store_true")
    args = parser.parse_args()
    try:
        config = load_config(args.config, args.require_ready)
        print(json.dumps({"configuration_count": len(expand_grid(config)),
                          "pending_settings": validate_config(config),
                          "grid": [r["model_hyperparameters"] for r in expand_grid(config)]}, indent=2))
    except (ValueError, OSError) as error:
        parser.exit(2, f"Configuration error: {error}\n")


### src/training/integration_check.py

In [ ]:
%%writefile /content/issue2_grid_project/src/training/integration_check.py
"""Verify frozen artifacts, then run a bounded real-data training/validation check."""
import argparse
import hashlib
import importlib.util
import json
import platform
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from src.models.sasrec import SASRec
from src.training.config import load_config

REQUIRED = ['dataset_info.json', 'rec_eval.py', 'sequences.parquet',
            'eval/eval_protocol.json', 'eval/neg_val.npy', 'eval/neg_test.npy',
            'eval/eval_cases.parquet', 'groups/item_groups.parquet', 'groups/user_groups.parquet']


def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def load_verified_data(path, config):
    path = Path(path)
    manifest = json.loads((path / 'manifest.json').read_text())
    checked = {}
    for name in REQUIRED:
        actual = digest(path / name)
        if actual != manifest['processed_files'][name]['sha256']:
            raise ValueError(f'Manifest checksum mismatch: {name}')
        checked[name] = actual
    # Execute only after checking against the team's manifest and reviewing source.
    spec = importlib.util.spec_from_file_location('shared_rec_eval', path / 'rec_eval.py')
    evaluator = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(evaluator)
    data = evaluator.load_dataset(path)
    info = data['info']
    expected = config['data']
    if (info['dataset'] != expected['expected_dataset'] or
        info['subset']['value'] != expected['expected_subset'] or
        info['max_seq_len'] != expected['max_seq_len'] or info['padding_idx'] != 0):
        raise ValueError('Dataset identity, subset, sequence length or padding mismatch')
    protocol = data['protocol']
    if (protocol['main'] != 'full_catalog_ranking' or not protocol['mask_history'] or
        10 not in protocol['ks']):
        raise ValueError('Expected full-catalog, history-masked NDCG@10 protocol')
    seq = data['sequences']
    if len(seq) != data['n_users'] or seq.user_idx.nunique() != len(seq):
        raise ValueError('User count or uniqueness mismatch')
    interactions = 0
    for row in seq.itertuples():
        history = list(row.train_items)
        ids = history + [row.val_item, row.test_item]
        if len(history) < 3 or any(i < 1 or i > data['n_items'] for i in ids):
            raise ValueError('Invalid training length or item IDs')
        if len(set(ids)) != len(ids):
            raise ValueError('Repeated items or holdout overlap')
        interactions += len(ids)
    if interactions != info['n_interactions']:
        raise ValueError('Interaction count mismatch')
    for key in ('neg_val', 'neg_test'):
        if data[key].shape[0] != len(seq):
            raise ValueError('Negative sample row count mismatch')
    return evaluator, data, checked


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--config', default='configs/config.json')
    parser.add_argument('--processed-dir', default='data/processed')
    parser.add_argument('--output-dir', default='results/issue1_smoke')
    parser.add_argument('--train-users', type=int, default=256)
    parser.add_argument('--val-users', type=int, default=128)
    parser.add_argument('--epochs', type=int, default=2)
    parser.add_argument('--batch-size', type=int, default=16)
    args = parser.parse_args()
    if min(args.train_users, args.val_users, args.epochs, args.batch_size) < 1:
        parser.error('Smoke-test counts must be positive')
    output = Path(args.output_dir)
    output.mkdir(parents=True, exist_ok=True)
    report_path = output / 'integration_report.json'
    if report_path.exists():
        parser.error('Output already contains a report; use a new output directory')
    config = load_config(args.config)
    evaluator, data, hashes = load_verified_data(args.processed_dir, config)
    random.seed(42); np.random.seed(42); torch.manual_seed(42)
    torch.set_num_threads(2)
    torch.use_deterministic_algorithms(True)
    # Explicit smoke settings, not silently promoted to full-training defaults.
    settings = dict(hidden_size=64, dropout=0.2, learning_rate=0.001,
                    num_heads=4, num_layers=2, weight_decay=1e-5, seed=42,
                    epochs=args.epochs, batch_size=args.batch_size, device='cpu')
    model = SASRec(data['n_items'], data['max_seq_len'], hidden_size=64,
                   num_heads=4, num_layers=2, dropout=0.2)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-5)
    seq = data['sequences'].iloc[:args.train_users]
    x, y = evaluator.train_arrays(seq, data['max_seq_len'])
    loader = DataLoader(TensorDataset(torch.from_numpy(x), torch.from_numpy(y)),
                        batch_size=args.batch_size, shuffle=True,
                        generator=torch.Generator().manual_seed(42))
    losses = []
    before = model.item_embedding.weight.detach().clone()
    start = time.perf_counter()
    for epoch in range(args.epochs):
        model.train()
        total, count = 0.0, 0
        for tokens, targets in loader:
            optimizer.zero_grad(set_to_none=True)
            logits, labels = model.training_logits(tokens, targets)
            loss = F.cross_entropy(logits, labels)
            if not torch.isfinite(loss):
                raise ValueError('Nonfinite training loss')
            loss.backward()
            norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
            optimizer.step()
            total += loss.item() * len(labels); count += len(labels)
        losses.append(total / count)
        print(f'epoch={epoch + 1} loss={losses[-1]:.6f}', flush=True)
    if torch.equal(before, model.item_embedding.weight):
        raise ValueError('Training did not update weights')

    @torch.no_grad()
    def score_fn(inputs, users):
        model.eval()
        scores = model.next_item_scores(torch.as_tensor(inputs)).numpy()
        if scores.shape != (len(inputs), data['n_items'] + 1) or not np.isfinite(scores).all():
            raise ValueError('Invalid score matrix')
        return scores

    users = data['sequences'].user_idx.iloc[:args.val_users].to_numpy()
    result = evaluator.evaluate(score_fn, data, split='val', sampled=False,
                                users=users, batch_size=args.batch_size)
    if not all(np.isfinite(v) for v in result['metrics'].values()):
        raise ValueError('Nonfinite validation metric')
    report = dict(status='PASS', purpose='integration_smoke_test_not_final_baseline',
                  settings=settings, train_users=len(seq), validation_users=result['n_cases'],
                  dataset_users=data['n_users'], catalog_items=data['n_items'],
                  split=result['split'], protocol=result['protocol'],
                  epoch_losses=losses, validation_metrics=result['metrics'],
                  test_evaluated=False, seconds=time.perf_counter()-start,
                  manifest_sha256=digest(Path(args.processed_dir)/'manifest.json'),
                  verified_files=hashes,
                  versions=dict(python=platform.python_version(), torch=torch.__version__,
                                numpy=np.__version__, pandas=pd.__version__))
    report_path.write_text(json.dumps(report, indent=2, allow_nan=False)+'\n')
    print(json.dumps(report, indent=2))

if __name__ == '__main__':
    main()


### src/training/trainer.py

In [ ]:
%%writefile /content/issue2_grid_project/src/training/trainer.py
"""Part C: one configuration/seed, validation-only selection, resumable training.

Model contract: training_logits(input_seq, target_seq) -> logits, nonzero labels;
next_item_scores(input_seq) -> (batch, catalog + padding) scores.
This module does not choose a model, launch a grid, or evaluate the test split.
"""
from copy import deepcopy
from dataclasses import asdict, dataclass
import csv
import io
import json
import math
import os
from pathlib import Path
import random
import tempfile
import time

import numpy as np
import torch
from torch.nn import functional as F


@dataclass(frozen=True)
class TrainingSettings:
    epochs: int
    patience: int
    optimizer: str
    learning_rate: float
    weight_decay: float
    seed: int
    device: str = 'cpu'
    min_delta: float = 0.0
    grad_clip_norm: float = 1.0
    lr_step_size: int | None = None
    lr_gamma: float | None = None

    def __post_init__(self):
        for key in ('epochs', 'patience'):
            value = getattr(self, key)
            if type(value) is not int or value < 1:
                raise ValueError(f'{key} must be a positive integer')
        if self.optimizer not in ('Adam', 'AdamW'):
            raise ValueError('optimizer must explicitly be Adam or AdamW')
        if type(self.seed) is not int or not 0 <= self.seed < 2**32:
            raise ValueError('seed must be an integer in [0, 2**32)')
        for key in ('learning_rate', 'weight_decay', 'min_delta', 'grad_clip_norm'):
            value = getattr(self, key)
            if type(value) not in (int, float) or not math.isfinite(value) or value < 0:
                raise ValueError(f'{key} must be finite and nonnegative')
        if self.learning_rate == 0 or self.grad_clip_norm == 0:
            raise ValueError('learning_rate and grad_clip_norm must be positive')
        if (self.lr_step_size is None) != (self.lr_gamma is None):
            raise ValueError('Specify both lr_step_size and lr_gamma, or neither')
        if self.lr_step_size is not None:
            if type(self.lr_step_size) is not int or self.lr_step_size < 1:
                raise ValueError('lr_step_size must be positive')
            if type(self.lr_gamma) not in (float, int) or not 0 < self.lr_gamma <= 1:
                raise ValueError('lr_gamma must be in (0, 1]')


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _atomic_write(path, write):
    path = Path(path)
    with tempfile.NamedTemporaryFile(dir=path.parent, prefix='.' + path.name,
                                     suffix='.tmp', delete=False) as temporary:
        name = Path(temporary.name)
    try:
        write(name)
        os.replace(name, path)
    finally:
        name.unlink(missing_ok=True)


def _json(path, value):
    text = json.dumps(value, indent=2, allow_nan=False) + '\n'
    _atomic_write(path, lambda temporary: temporary.write_text(text))


def _rng_state(loader):
    state = np.random.get_state()
    return {'python': random.getstate(), 'numpy': (state[0], state[1].tolist(), *state[2:]),
            'torch': torch.get_rng_state(),
            'cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
            'loader': loader.generator.get_state() if loader.generator is not None else None}


def _restore_rng(state, loader):
    random.setstate(state['python'])
    name, values, *rest = state['numpy']
    np.random.set_state((name, np.asarray(values, dtype=np.uint32), *rest))
    torch.set_rng_state(state['torch'].cpu())
    if state['cuda'] and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([s.cpu() for s in state['cuda']])
    if state['loader'] is not None:
        if loader.generator is None:
            raise ValueError('Resume requires the saved DataLoader generator')
        loader.generator.set_state(state['loader'].cpu())


def train_epoch(model, loader, optimizer, device, grad_clip_norm):
    model.train()
    total_loss, n_targets = 0.0, 0
    for batch in loader:
        x = batch['input_seq'].to(device)
        y = batch['target_seq'].to(device)
        if x.dtype != torch.long or y.dtype != torch.long or x.shape != y.shape or x.ndim != 2:
            raise ValueError('Expected matching int64 (batch, length) input_seq/target_seq')
        if not torch.equal(x.eq(0), y.eq(0)) or (x < 0).any() or (y < 0).any():
            raise ValueError('Invalid IDs or misaligned padding')
        keep = y.ne(0)
        if not keep.any():
            continue
        optimizer.zero_grad(set_to_none=True)
        logits, labels = model.training_logits(x, y)
        if logits.ndim != 2 or len(logits) != int(keep.sum()) or not torch.equal(labels, y[keep]):
            raise ValueError('training_logits must cover exactly the nonpadding target positions')
        if not torch.isfinite(logits).all():
            raise ValueError('Nonfinite training logits')
        loss = F.cross_entropy(logits, labels)
        if not torch.isfinite(loss):
            raise ValueError('Nonfinite loss')
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm, error_if_nonfinite=True)
        optimizer.step()
        total_loss += float(loss.detach()) * len(labels)
        n_targets += len(labels)
    if not n_targets:
        raise ValueError('No nonpadding training targets')
    return total_loss / n_targets, n_targets


def checked_metrics(metrics):
    if 'NDCG@10' not in metrics:
        raise ValueError('Validation must return NDCG@10')
    clean = {key: float(value) for key, value in metrics.items()}
    if not all(math.isfinite(v) and 0 <= v <= 1 for v in clean.values()):
        raise ValueError('Ranking metrics must be finite and in [0, 1]')
    return clean


def make_shared_validator(evaluator, data, *, device='cpu', batch_size=128, users=None):
    """Always call the frozen rec_eval.py on full-catalog validation, never test."""
    @torch.no_grad()
    def validate(model):
        model.eval()

        def score_fn(inputs, user_ids):
            del user_ids
            scores = model.next_item_scores(torch.as_tensor(inputs, dtype=torch.long, device=device))
            if scores.shape != (len(inputs), data['n_items'] + 1) or not torch.isfinite(scores).all():
                raise ValueError('Invalid/nonfinite score matrix before history masking')
            return scores.detach().cpu().numpy()

        result = evaluator.evaluate(score_fn, data, split='val', sampled=False,
                                    users=users, batch_size=batch_size, return_topk=False)
        ranks = np.asarray(result['ranks'])
        if not len(ranks) or not np.isfinite(ranks).all() or (ranks < 1).any():
            raise ValueError('Invalid validation ranks')
        return checked_metrics(result['metrics'])
    return validate


def fit(model, train_loader, validate, settings, output_dir, *, metadata, resume_from=None):
    """Train one run. Resume is epoch-boundary, num_workers=0 only, into a NEW directory.

    Epoch budget may be extended on resume; other settings and metadata must match.
    Only load checkpoints produced by this project from a trusted location.
    """
    if train_loader.num_workers != 0:
        raise ValueError('Use num_workers=0 for reproducible epoch-boundary resume')
    output = Path(output_dir)
    if output.exists():
        raise FileExistsError('Run directory already exists; choose a new output directory')
    config = asdict(settings)
    # Require serializable provenance before creating any run artifacts.
    json.dumps(metadata, allow_nan=False)
    model.to(settings.device)
    optimizer = getattr(torch.optim, settings.optimizer)(
        model.parameters(), lr=settings.learning_rate, weight_decay=settings.weight_decay)
    scheduler = (torch.optim.lr_scheduler.StepLR(optimizer, settings.lr_step_size, settings.lr_gamma)
                 if settings.lr_step_size is not None else None)
    history, start_epoch, bad_epochs = [], 0, 0
    best_score, best_epoch, best_state = -math.inf, 0, None
    stopping_score = -math.inf
    if resume_from:
        checkpoint = torch.load(resume_from, map_location=settings.device, weights_only=True)
        if checkpoint['schema_version'] != 1 or checkpoint['metadata'] != metadata:
            raise ValueError('Checkpoint provenance/model/data mismatch')
        for key, value in config.items():
            if key not in ('epochs', 'device') and checkpoint['settings'][key] != value:
                raise ValueError(f'Checkpoint setting mismatch: {key}')
        start_epoch = checkpoint['epoch']
        if settings.epochs <= start_epoch or checkpoint['bad_epochs'] >= settings.patience:
            raise ValueError('Checkpoint is complete or already early-stopped')
        model.load_state_dict(checkpoint['model'])
        optimizer.load_state_dict(checkpoint['optimizer'])
        if scheduler is not None:
            scheduler.load_state_dict(checkpoint['scheduler'])
        history, bad_epochs = checkpoint['history'], checkpoint['bad_epochs']
        best_score, best_epoch = checkpoint['best_score'], checkpoint['best_epoch']
        best_state = checkpoint['best_model']
        stopping_score = checkpoint['stopping_score']
        _restore_rng(checkpoint['rng'], train_loader)
    else:
        seed_everything(settings.seed)
    # Atomic ownership: two jobs must never both write the same run directory.
    output.mkdir(parents=True, exist_ok=False)
    _json(output / 'run_config.json', {'training': config, 'metadata': metadata,
                                     'resume_from': str(resume_from) if resume_from else None})
    try:
        for epoch in range(start_epoch + 1, settings.epochs + 1):
            begin = time.perf_counter()
            learning_rate = optimizer.param_groups[0]['lr']
            loss, count = train_epoch(model, train_loader, optimizer, settings.device,
                                       settings.grad_clip_norm)
            train_seconds = time.perf_counter() - begin
            model.eval()
            with torch.no_grad():
                metrics = checked_metrics(validate(model))
            improved = metrics['NDCG@10'] > best_score
            if improved:
                best_score, best_epoch = metrics['NDCG@10'], epoch
                best_state = deepcopy(model.state_dict())
            # min_delta controls stopping only, not which checkpoint is best.
            if metrics['NDCG@10'] > stopping_score + settings.min_delta:
                stopping_score, bad_epochs = metrics['NDCG@10'], 0
            else:
                bad_epochs += 1
            if scheduler is not None:
                scheduler.step()
            history.append({'epoch': epoch, 'train_loss': loss, 'train_targets': count,
                            'learning_rate': learning_rate, 'seconds': time.perf_counter() - begin,
                            'train_seconds': train_seconds,
                            'validation_seconds': time.perf_counter() - begin - train_seconds,
                            'improved': improved, **{'val_' + k: v for k, v in metrics.items()}})
            stream = io.StringIO()
            writer = csv.DictWriter(stream, fieldnames=list(history[0]))
            writer.writeheader()
            writer.writerows(history)
            _atomic_write(output / 'history.csv', lambda p: p.write_text(stream.getvalue()))
            checkpoint = {'schema_version': 1, 'epoch': epoch, 'model': model.state_dict(),
                          'optimizer': optimizer.state_dict(),
                          'scheduler': scheduler.state_dict() if scheduler is not None else None,
                          'rng': _rng_state(train_loader), 'settings': config, 'metadata': metadata,
                          'history': history, 'bad_epochs': bad_epochs, 'best_score': best_score,
                          'stopping_score': stopping_score,
                          'best_epoch': best_epoch, 'best_model': best_state}
            _atomic_write(output / 'last.pt', lambda p: torch.save(checkpoint, p))
            if improved or not (output / 'best.pt').exists():
                _atomic_write(output / 'best.pt', lambda p: torch.save(
                    {'schema_version': 1, 'model': best_state, 'epoch': best_epoch,
                     'validation_NDCG@10': best_score, 'metadata': metadata}, p))
            print(f'epoch={epoch} loss={loss:.6f} val_NDCG@10={metrics["NDCG@10"]:.6f}', flush=True)
            if bad_epochs >= settings.patience:
                break
        summary = {'status': 'PASS', 'epochs_completed': history[-1]['epoch'],
                   'best_epoch': best_epoch, 'best_validation_NDCG@10': best_score,
                   'early_stopped': bad_epochs >= settings.patience, 'test_evaluated': False,
                   'checkpoint': 'best.pt', 'metadata': metadata}
        _json(output / 'summary.json', summary)
        return summary
    except Exception as error:
        _json(output / 'failure.json', {'status': 'FAILED', 'type': type(error).__name__,
                                       'message': str(error), 'test_evaluated': False})
        raise


def load_best(model, checkpoint_path, *, device='cpu'):
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
    model.to(device).load_state_dict(checkpoint['model'])
    model.eval()
    return checkpoint


### src/training/train.py

In [ ]:
%%writefile /content/issue2_grid_project/src/training/train.py
"""Part C entry point: run ONE configuration and ONE seed, not a grid search."""
import argparse
import json
from pathlib import Path
import platform
import subprocess

import numpy as np
import pandas as pd
import torch

from src.data.sequence_dataset import make_dataloaders
from src.models.sasrec import SASRec
from src.training.config import GRID, expand_grid, validate_config
from src.training.integration_check import digest, load_verified_data
from src.training.trainer import (TrainingSettings, fit, load_best,
                                  make_shared_validator, seed_everything)

TIE_POLICY = {
    'name': 'average_rank_then_metrics',
    'status': 'team_accepted_frozen_convention',
    'definition': 'rank = 1 + num_greater + num_equal_others / 2; metrics use that rank',
    'limitation': 'Not the expected metrics over random orderings of tied items',
}


def resolve_config(path, grid_index, seed):
    """Accept either Part D's grid config or one dict from expand_grid()."""
    config = json.loads(Path(path).read_text())
    if 'search' in config:
        validate_config(config, require_ready=True)
        if grid_index is None or not 0 <= grid_index < 8:
            raise ValueError('Select exactly one configuration with --grid-index 0..7')
        config = expand_grid(config)[grid_index]
    else:
        if grid_index is not None:
            raise ValueError('Expanded configurations do not take --grid-index')
        validate_config({**config, 'search': GRID}, require_ready=True)
        parameters = config.get('model_hyperparameters', {})
        if set(parameters) != set(GRID) or any(parameters[k] not in GRID[k] for k in GRID):
            raise ValueError('Invalid expanded model_hyperparameters')
    if seed not in config['training']['seeds']:
        raise ValueError('--seed must be one of the explicitly configured seeds')
    return config


def code_provenance():
    root = Path(__file__).resolve().parents[2]
    files = ['src/data/sequence_dataset.py', 'src/models/sasrec.py',
             'src/training/trainer.py', 'src/training/train.py',
             'src/training/config.py', 'src/training/integration_check.py']
    try:
        commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=root, text=True).strip()
    except (OSError, subprocess.CalledProcessError):
        commit = None
    return {'commit': commit, 'source_sha256': {name: digest(root / name) for name in files},
            'versions': {'python': platform.python_version(), 'torch': str(torch.__version__),
                         'numpy': np.__version__, 'pandas': pd.__version__}}


def synthetic_smoke(output_dir):
    """A tiny CPU fixture, not Amazon data or the project's research evaluator."""
    torch.set_num_threads(2)
    seed_everything(42)
    sequences = pd.DataFrame({
        'user_idx': list(range(8)),
        'train_items': [[u + 1, u + 9, u + 17] for u in range(8)],
        'val_item': [25 + u % 3 for u in range(8)],
        'test_item': [28 + u % 3 for u in range(8)],
    })
    loaders = make_dataloaders(sequences, max_len=50, batch_size=4,
                               eval_batch_size=4, seed=42, pin_memory=False)
    model = SASRec(30, max_len=50, hidden_size=16, num_heads=2, num_layers=1, dropout=0.1)

    @torch.no_grad()
    def validate(current):
        # Synthetic-only deterministic item-ID tie breaking. This is NOT a new
        # evaluator for the Amazon benchmark; real runs call frozen rec_eval.py.
        ranks = []
        for batch in loaders['val']:
            scores = current.next_item_scores(batch['input_seq']).clone()
            if not torch.isfinite(scores).all():
                raise ValueError('Nonfinite synthetic scores')
            scores[:, 0] = -torch.inf
            scores.scatter_(1, batch['history'], -torch.inf)
            targets = batch['target']
            target_scores = scores.gather(1, targets[:, None])
            item_ids = torch.arange(scores.shape[1])[None, :]
            rank = 1 + (scores > target_scores).sum(1) + (
                (scores == target_scores) & (item_ids < targets[:, None])).sum(1)
            ranks.extend(rank.tolist())
        ranks = np.asarray(ranks, dtype=float)
        return {'NDCG@10': float(np.where(ranks <= 10, 1 / np.log2(ranks + 1), 0).mean()),
                'Recall@10': float((ranks <= 10).mean()), 'MRR': float((1 / ranks).mean())}

    settings = TrainingSettings(epochs=3, patience=2, optimizer='Adam', learning_rate=0.001,
                                 weight_decay=0.0, seed=42)
    metadata = {'purpose': 'synthetic_smoke_only_not_Amazon_results', 'users': 8,
                'items': 30, 'selection_split': 'val', **code_provenance()}
    summary = fit(model, loaders['train'], validate, settings, output_dir, metadata=metadata)
    restored = SASRec(30, max_len=50, hidden_size=16, num_heads=2, num_layers=1, dropout=0.1)
    checkpoint = load_best(restored, Path(output_dir) / 'best.pt')
    if checkpoint['epoch'] != summary['best_epoch']:
        raise ValueError('Checkpoint epoch mismatch')
    restored_metrics = validate(restored)
    if restored_metrics['NDCG@10'] != summary['best_validation_NDCG@10']:
        raise ValueError('Reloaded checkpoint validation mismatch')
    print('Synthetic checkpoint reload: PASS (not formal Model 0 results)')
    return summary


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--config', default='configs/config.json')
    parser.add_argument('--grid-index', type=int)
    parser.add_argument('--seed', type=int)
    parser.add_argument('--optimizer', choices=['Adam', 'AdamW'])
    parser.add_argument('--device', choices=['cpu', 'cuda', 'mps'], default='cpu')
    parser.add_argument('--output-dir', required=True, help='New run directory; never overwrite a run')
    parser.add_argument('--resume', help='Trusted last.pt, resuming into a new output directory')
    parser.add_argument('--eval-batch-size', type=int, default=128)
    parser.add_argument('--limit-train-users', type=int)
    parser.add_argument('--limit-val-users', type=int)
    parser.add_argument('--min-delta', type=float, default=0.0)
    parser.add_argument('--lr-step-size', type=int)
    parser.add_argument('--lr-gamma', type=float)
    parser.add_argument('--synthetic-smoke', action='store_true')
    args = parser.parse_args()
    if args.synthetic_smoke:
        if args.resume:
            parser.error('The synthetic CLI smoke is standalone; resume is covered by unit tests')
        synthetic_smoke(args.output_dir)
        return
    try:
        config = resolve_config(args.config, args.grid_index, args.seed)
        if args.optimizer is None:
            raise ValueError('Choose --optimizer explicitly; Adam and AdamW are not interchangeable')
        if args.eval_batch_size < 1 or any(v is not None and v < 1 for v in
                                         (args.limit_train_users, args.limit_val_users)):
            raise ValueError('Batch size and optional user limits must be positive')
        training, hyper = config['training'], config['model_hyperparameters']
        if training.get('optimizer', args.optimizer) != args.optimizer:
            raise ValueError('CLI optimizer differs from configured optimizer')
        if training.get('scheduler') == 'none' and (args.lr_step_size is not None or args.lr_gamma is not None):
            raise ValueError('Configured scheduler=none forbids scheduler overrides')
        settings = TrainingSettings(
            epochs=training['epochs'], patience=training['early_stopping_patience'],
            optimizer=args.optimizer, learning_rate=hyper['learning_rate'],
            weight_decay=training['weight_decay'], seed=args.seed, device=args.device,
            min_delta=args.min_delta, lr_step_size=args.lr_step_size, lr_gamma=args.lr_gamma)
        evaluator, data, hashes = load_verified_data(config['data']['processed_dir'], config)
        sequences = data['sequences'].iloc[:args.limit_train_users] if args.limit_train_users else data['sequences']
        users = data['sequences'].user_idx.iloc[:args.limit_val_users].to_numpy()
        seed_everything(args.seed)
        model = SASRec(data['n_items'], max_len=data['max_seq_len'], hidden_size=hyper['hidden_size'],
                       dropout=hyper['dropout'], num_heads=training['num_heads'],
                       num_layers=training['num_layers'])
        loaders = make_dataloaders(sequences, max_len=data['max_seq_len'],
                                   batch_size=training['batch_size'], seed=args.seed,
                                   sliding_window=False, pin_memory=args.device == 'cuda')
        validate = make_shared_validator(evaluator, data, device=args.device,
                                          batch_size=args.eval_batch_size, users=users)
        metadata = {'purpose': 'model0_single_config_training',
                    'selection_split': 'val', 'selection_metric': 'NDCG@10', 'sampled': False,
                    'legacy_tie_policy_unresolved': False, 'tie_policy': TIE_POLICY.copy(),
                    'test_evaluated': False,
                    'model_hyperparameters': hyper,
                    'model_layers': training['num_layers'], 'model_heads': training['num_heads'],
                    'max_seq_len': data['max_seq_len'], 'batch_size': training['batch_size'],
                    'train_users': len(sequences), 'validation_users': len(users),
                    'eval_batch_size': args.eval_batch_size, 'sliding_window': False,
                    'manifest_sha256': digest(Path(config['data']['processed_dir']) / 'manifest.json'),
                    'verified_files': hashes, **code_provenance()}
        fit(model, loaders['train'], validate, settings, args.output_dir,
             metadata=metadata, resume_from=args.resume)
    except (ValueError, FileNotFoundError, FileExistsError) as error:
        parser.exit(2, f'Training not started/completed: {error}\n')


if __name__ == '__main__':
    main()


### src/training/grid_search.py

In [ ]:
%%writefile /content/issue2_grid_project/src/training/grid_search.py
"""Part D: run all eight configurations using Part C, on one explicit tuning seed."""
import argparse
import csv
import json
import math
from pathlib import Path
import shutil
import subprocess
import sys
import time

from src.training.config import expand_grid, load_config
from src.training.integration_check import digest
from src.training.trainer import _json
from src.training.train import TIE_POLICY

ROOT = Path(__file__).resolve().parents[2]


def run_grid(config_path, output_dir, *, seed, optimizer, device='cpu',
             eval_batch_size=128, limit_train_users=None, limit_val_users=None,
             discard_last_checkpoint=False):
    config = load_config(config_path, require_ready=True)
    if config['training'].get('optimizer', optimizer) != optimizer:
        raise ValueError('Optimizer differs from the configured optimizer')
    if seed not in config['training']['seeds']:
        raise ValueError('Tuning seed must be in training.seeds')
    if optimizer not in ('Adam', 'AdamW') or device not in ('cpu', 'cuda', 'mps'):
        raise ValueError('Choose a supported optimizer and device explicitly')
    if any(type(v) is not int or v < 1 for v in
           [eval_batch_size] + [v for v in (limit_train_users, limit_val_users) if v is not None]):
        raise ValueError('Batch size and user limits must be positive integers')
    # Resolve the data path before child processes change to the repository root.
    config['data']['processed_dir'] = str(Path(config['data']['processed_dir']).resolve())
    output = Path(output_dir).resolve()
    output.mkdir(parents=True, exist_ok=False)
    snapshot = output / 'search_config.json'
    _json(snapshot, config)
    plan = {'purpose': 'model0_grid_search',
            'user_limited': limit_train_users is not None or limit_val_users is not None,
            'tie_policy': TIE_POLICY.copy(),
            'selection_split': 'val', 'selection_metric': 'NDCG@10', 'sampled': False,
            'seed': seed, 'seed_policy': 'one_explicit_tuning_seed',
            'optimizer': optimizer, 'device': device, 'eval_batch_size': eval_batch_size,
            'discard_last_checkpoint': discard_last_checkpoint,
            'limit_train_users': limit_train_users, 'limit_val_users': limit_val_users,
            'tie_break': 'first_configuration_in_config_order', 'test_evaluated': False,
            'runner_sha256': digest(Path(__file__)), 'configuration_count': 8}
    _json(output / 'search_plan.json', plan)
    rows = []
    try:
        for index, expanded in enumerate(expand_grid(config)):
            run_dir = output / f'run_{index:02d}_seed_{seed}'
            command = [sys.executable, '-B', '-m', 'src.training.train',
                       '--config', str(snapshot), '--grid-index', str(index),
                       '--seed', str(seed), '--optimizer', optimizer, '--device', device,
                       '--eval-batch-size', str(eval_batch_size), '--output-dir', str(run_dir)]
            for flag, value in (('--limit-train-users', limit_train_users),
                                ('--limit-val-users', limit_val_users)):
                if value is not None:
                    command.extend([flag, str(value)])
            row = {'grid_index': index, 'seed': seed, **expanded['model_hyperparameters'],
                   'status': 'FAILED', 'validation_NDCG@10': '', 'best_epoch': '',
                   'seconds': 0, 'run_dir': run_dir.name}
            start = time.perf_counter()
            print(f'Running configuration {index + 1}/8: {expanded["model_hyperparameters"]}', flush=True)
            try:
                with (output / f'run_{index:02d}.log').open('w') as log:
                    subprocess.run(command, cwd=ROOT, stdout=log, stderr=subprocess.STDOUT, check=True)
                summary = json.loads((run_dir / 'summary.json').read_text())
                score = summary['best_validation_NDCG@10']
                metadata = summary['metadata']
                if (summary['status'] != 'PASS' or summary['test_evaluated'] is not False
                    or metadata['selection_split'] != 'val' or metadata['sampled'] is not False
                    or metadata.get('tie_policy') != TIE_POLICY
                    or metadata['model_hyperparameters'] != expanded['model_hyperparameters']
                    or type(score) not in (int, float) or not math.isfinite(score) or not 0 <= score <= 1
                    or not (run_dir / 'best.pt').is_file()):
                    raise ValueError('Invalid validation result or missing checkpoint')
                row.update({'status': 'PASS', 'validation_NDCG@10': score})
                row['best_epoch'] = summary['best_epoch']
                if discard_last_checkpoint:
                    (run_dir / 'last.pt').unlink(missing_ok=True)
            finally:
                row['seconds'] = time.perf_counter() - start
                rows.append(row)
                with (output / 'grid_results.csv').open('w', newline='') as stream:
                    writer = csv.DictWriter(stream, fieldnames=list(row))
                    writer.writeheader()
                    writer.writerows(rows)
        # max preserves configuration order on equal scores; never select partial grids.
        best = max(rows, key=lambda r: r['validation_NDCG@10'])
        selected = expand_grid(config)[best['grid_index']]
        selected['training']['seeds'] = [seed]
        selected['output_dir'] = str(output / best['run_dir'])
        shutil.copyfile(output / best['run_dir'] / 'best.pt', output / 'best_checkpoint.pt')
        _json(output / 'best_config.json', selected)
        result = {**plan, 'status': 'PASS', 'completed_runs': len(rows),
                  'best_grid_index': best['grid_index'],
                  'best_validation_NDCG@10': best['validation_NDCG@10'],
                  'best_epoch': best['best_epoch'], 'source_checkpoint': best['run_dir'] + '/best.pt',
                  'checkpoint_sha256': digest(output / 'best_checkpoint.pt')}
        _json(output / 'selection.json', result)
        return result
    except Exception as error:
        _json(output / 'failure.json', {'status': 'FAILED', 'error': str(error),
                                       'completed_runs': sum(r['status'] == 'PASS' for r in rows),
                                       'test_evaluated': False})
        raise


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--config', default='configs/config.json')
    parser.add_argument('--output-dir')
    parser.add_argument('--seed', type=int)
    parser.add_argument('--optimizer', choices=['Adam', 'AdamW'])
    parser.add_argument('--device', choices=['cpu', 'cuda', 'mps'], default='cpu')
    parser.add_argument('--eval-batch-size', type=int, default=128)
    parser.add_argument('--limit-train-users', type=int)
    parser.add_argument('--limit-val-users', type=int)
    parser.add_argument('--discard-last-checkpoint', action='store_true',
                        help='Delete each successful run resume checkpoint; keep best.pt')
    parser.add_argument('--dry-run', action='store_true')
    args = parser.parse_args()
    try:
        if args.dry_run:
            print(json.dumps(expand_grid(load_config(args.config)), indent=2))
            return
        if args.seed is None or args.optimizer is None or args.output_dir is None:
            parser.error('--seed, --optimizer and --output-dir are required for execution')
        options = vars(args).copy()
        options.pop('dry_run')
        options['config_path'] = options.pop('config')
        print(json.dumps(run_grid(**options), indent=2))
    except (ValueError, OSError, subprocess.CalledProcessError) as error:
        parser.exit(2, f'Grid search failed: {error}\n')


if __name__ == '__main__':
    main()


### src/training/checkpoints.py

In [ ]:
%%writefile /content/issue2_grid_project/src/training/checkpoints.py
"""Load a trusted Part D winner with configuration and checksum verification.

Part C owns checkpoint writing and epoch-boundary resume. This adapter restores
an inference model from the three portable selection artifacts, without data access.
"""
import argparse
import json
import math
from pathlib import Path

import torch

from src.models.sasrec import SASRec
from src.training.config import GRID, validate_config
from src.training.integration_check import digest


def load_selected_model(directory, device='cpu'):
    """Return (eval-mode model, checkpoint, selection) from a trusted run directory."""
    directory = Path(directory)
    selection = json.loads((directory / 'selection.json').read_text())
    config = json.loads((directory / 'best_config.json').read_text())
    checkpoint_path = directory / 'best_checkpoint.pt'
    if (selection.get('status') != 'PASS' or selection.get('completed_runs') != 8
            or selection.get('selection_split') != 'val'
            or selection.get('selection_metric') != 'NDCG@10'
            or selection.get('sampled') is not False or selection.get('test_evaluated') is not False):
        raise ValueError('Expected a complete validation-only grid selection')
    if digest(checkpoint_path) != selection.get('checkpoint_sha256'):
        raise ValueError('Selected checkpoint checksum mismatch')
    validate_config({**config, 'search': GRID}, require_ready=True)
    hyper = config['model_hyperparameters']
    if set(hyper) != set(GRID) or any(hyper[k] not in GRID[k] for k in GRID):
        raise ValueError('Invalid selected hyperparameters')
    if config['training']['seeds'] != [selection['seed']]:
        raise ValueError('Selected seed mismatch')
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    if checkpoint.get('schema_version') != 1:
        raise ValueError('Unsupported checkpoint schema')
    score = checkpoint['validation_NDCG@10']
    if (not math.isfinite(score) or not 0 <= score <= 1
            or score != selection['best_validation_NDCG@10']
            or checkpoint['epoch'] != selection['best_epoch']):
        raise ValueError('Selected epoch/validation score mismatch')
    metadata = checkpoint['metadata']
    expected = {'model_hyperparameters': hyper,
                'model_layers': config['training']['num_layers'],
                'model_heads': config['training']['num_heads'],
                'max_seq_len': config['data']['max_seq_len'],
                'selection_split': 'val', 'selection_metric': 'NDCG@10', 'sampled': False,
                'tie_policy': selection['tie_policy']}
    if any(metadata.get(k) != v for k, v in expected.items()):
        raise ValueError('Checkpoint metadata does not match selected configuration/protocol')
    n_items = checkpoint['model']['item_embedding.weight'].shape[0] - 1
    model = SASRec(n_items, max_len=metadata['max_seq_len'], hidden_size=hyper['hidden_size'],
                   dropout=hyper['dropout'], num_heads=metadata['model_heads'],
                   num_layers=metadata['model_layers'])
    model.load_state_dict(checkpoint['model'], strict=True)
    model.to(device).eval()
    return model, checkpoint, selection


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--run-dir', required=True)
    parser.add_argument('--device', choices=['cpu', 'cuda', 'mps'], default='cpu')
    args = parser.parse_args()
    model, checkpoint, selection = load_selected_model(args.run_dir, args.device)
    print(json.dumps({'status': 'PASS', 'epoch': checkpoint['epoch'],
                      'validation_NDCG@10': selection['best_validation_NDCG@10'],
                      'model_in_eval_mode': not model.training, 'test_evaluated': False}))


if __name__ == '__main__':
    main()


### src/training/preflight.py

In [ ]:
%%writefile /content/issue2_grid_project/src/training/preflight.py
"""Verify frozen data and initial SASRec losses before the agreed grid search."""
import argparse
import json
import math
from pathlib import Path
import platform
import torch
from src.models.sasrec import SASRec
from src.data.sequence_dataset import make_dataloaders
from src.training.config import load_config
from src.training.integration_check import load_verified_data,digest
from src.training.trainer import seed_everything
from src.training.train import code_provenance,TIE_POLICY


def check(config_path, output):
    if Path(output).exists():
        raise FileExistsError('Choose a new preflight report path')
    config=load_config(config_path,require_ready=True)
    evaluator,data,hashes=load_verified_data(config['data']['processed_dir'],config)
    training=config['training']
    torch.set_num_threads(2)
    rows=[]
    for hidden in config['search']['hidden_size']:
        seed_everything(42)
        model=SASRec(data['n_items'],max_len=50,hidden_size=hidden,
                     num_heads=training['num_heads'],num_layers=training['num_layers'],dropout=.2).eval()
        loader=make_dataloaders(data['sequences'].iloc[:128],max_len=50,batch_size=128,
                                seed=42,pin_memory=False)['train']
        batch=next(iter(loader))
        with torch.no_grad():
            logits,labels=model.training_logits(batch['input_seq'],batch['target_seq'])
            loss=torch.nn.functional.cross_entropy(logits,labels).item()
        reference=math.log(data['n_items']+1)
        if not math.isfinite(loss) or abs(loss-reference)>.5:
            raise ValueError(f'Unexpected initial loss for hidden size {hidden}: {loss}')
        if torch.count_nonzero(model.item_embedding.weight[0]):
            raise ValueError('Padding row is not zero at initialization')
        rows.append({'hidden_size':hidden,'initial_loss':loss,'uniform_reference':reference})
    report={'status':'PASS','purpose':'initialization_and_data_check_not_final_results',
            'checks':rows,'verified_files':hashes,'tie_policy':TIE_POLICY,'test_evaluated':False,
            'config_sha256':digest(Path(config_path)),**code_provenance()}
    Path(output).parent.mkdir(parents=True,exist_ok=True)
    with Path(output).open('x') as stream: json.dump(report,stream,indent=2,allow_nan=False)
    print(json.dumps(report['checks'],indent=2))
    return report

if __name__=='__main__':
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--config',default='configs/config.json')
    parser.add_argument('--output',required=True)
    args=parser.parse_args()
    check(args.config,args.output)


### Verify the declared environment before loading data


In [ ]:
%%writefile /content/issue2_grid_project/src/training/environment_check.py
"""Fail before training when the runtime differs from an explicit environment target."""
import argparse
from datetime import datetime, timezone
import importlib
import json
from pathlib import Path
import platform
import subprocess
import sys


def collect_environment():
    versions = {'python': platform.python_version()}
    modules = {}
    for name in ('torch', 'numpy', 'pandas', 'pyarrow'):
        modules[name] = importlib.import_module(name)
        versions[name] = str(modules[name].__version__)
    torch = modules['torch']
    cuda = torch.cuda.is_available()
    return {'versions': versions, 'cuda_available': cuda,
            'cuda_build': torch.version.cuda,
            'cudnn_version': torch.backends.cudnn.version(),
            'gpu_names': [torch.cuda.get_device_name(i)
                          for i in range(torch.cuda.device_count())] if cuda else [],
            'platform': platform.platform(), 'python_executable': sys.executable}


def environment_mismatches(actual, expected):
    required = {'python', 'torch', 'numpy', 'pandas', 'pyarrow'}
    if set(expected.get('versions', {})) != required:
        raise ValueError('Environment target must specify Python and all four library versions')
    if not all(isinstance(v, str) and v for v in expected['versions'].values()):
        raise ValueError('Environment target versions must be nonempty strings')
    if type(expected.get('require_cuda')) is not bool:
        raise ValueError('Environment target must declare require_cuda as a boolean')
    found = [f'{name}: actual {actual.get("versions", {}).get(name)!r}, expected {version!r}'
             for name, version in expected['versions'].items()
             if actual.get('versions', {}).get(name) != version]
    if expected['require_cuda'] and actual.get('cuda_available') is not True:
        found.append('CUDA is required but unavailable')
    return found


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--spec', default='configs/colab_environment.json')
    parser.add_argument('--output', required=True)
    args = parser.parse_args()
    expected = json.loads(Path(args.spec).read_text())
    actual = collect_environment()
    mismatches = environment_mismatches(actual, expected)
    report = {'status': 'FAIL' if mismatches else 'PASS',
              'checked_utc': datetime.now(timezone.utc).isoformat(),
              'expected': expected, 'actual': actual, 'mismatches': mismatches,
              'pip_freeze': subprocess.check_output(
                  [sys.executable, '-m', 'pip', 'freeze'], text=True).splitlines()}
    Path(args.output).write_text(json.dumps(report, indent=2) + '\n')
    print(json.dumps(report, indent=2))
    if mismatches:
        parser.exit(2, 'Environment mismatch: stop before training. Restore the declared runtime '
                    'or agree on a new environment target for a separately recorded experiment. '
                    'Do not mix environments across grid/repeat runs.\n')


if __name__ == '__main__':
    main()


In [ ]:
import json
ENVIRONMENT_SPEC = {'profile': 'model0_colab_20261107', 'versions': {'python': '3.13.16', 'torch': '2.14.1+cu130', 'numpy': '2.2.6', 'pandas': '2.3.2', 'pyarrow': '21.0.0'}, 'require_cuda': True, 'note': 'Version preflight target from the recorded full-grid runtime; not an installable lockfile or a guarantee of identical GPU results.'}
(PROJECT / "configs/colab_environment.json").write_text(json.dumps(ENVIRONMENT_SPEC, indent=2) + "\n")
subprocess.run([sys.executable, "-B", "-m", "src.training.environment_check",
    "--spec", str(PROJECT / "configs/colab_environment.json"),
    "--output", str(PROJECT / "environment_preflight.json")], cwd=PROJECT, check=True)


## 4. Download and verify the prepared data

This reads `processed` inside your shared `NewDataRunFiles` folder by Drive ID.
Authorize with the account that has access. No shortcut or mounted input path is
needed. Only the ten required files are downloaded to runtime storage; the shared
folder is not changed. Nine file hashes are checked against the team manifest
before the evaluator is imported.

Uses the official [Drive download API](https://developers.google.com/workspace/drive/api/guides/manage-downloads).


In [ ]:
import shutil
import json
import hashlib
import re
from google.colab import auth
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from googleapiclient.errors import HttpError

# Choose the Google account that can open the shared folder.
auth.authenticate_user()
credentials, _ = google.auth.default()
drive_api = build("drive", "v3", credentials=credentials, cache_discovery=False)
if not re.fullmatch(r"[A-Za-z0-9_-]+", DRIVE_DATA_FOLDER_ID):
    raise ValueError("DRIVE_DATA_FOLDER_ID must be the folder ID, not the whole URL.")

folder_cache = {}
def children(folder_id):
    if folder_id not in folder_cache:
        items, token = [], None
        while True:
            page = drive_api.files().list(
                q=f"'{folder_id}' in parents and trashed = false",
                fields="nextPageToken,files(id,name,mimeType,capabilities(canDownload))",
                pageSize=1000, pageToken=token,
                supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
            items.extend(page.get("files", []))
            token = page.get("nextPageToken")
            if not token:
                break
        folder_cache[folder_id] = items
    return folder_cache[folder_id]

def child(folder_id, name):
    matches = [entry for entry in children(folder_id) if entry["name"] == name]
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one '{name}', found {len(matches)}. Check the folder and signed-in account.")
    return matches[0]

required = ["dataset_info.json", "rec_eval.py", "sequences.parquet",
            "eval/eval_protocol.json", "eval/neg_val.npy", "eval/neg_test.npy",
            "eval/eval_cases.parquet", "groups/item_groups.parquet",
            "groups/user_groups.parquet"]
processed = PROJECT / "data/processed"
processed.mkdir(parents=True, exist_ok=True)
try:
    remote_processed = child(DRIVE_DATA_FOLDER_ID, "processed")
    if remote_processed["mimeType"] != "application/vnd.google-apps.folder":
        raise ValueError("The processed entry must be the actual shared folder.")
    for relative in ["manifest.json", *required]:
        folder_id = remote_processed["id"]
        parts = relative.split("/")
        for part in parts[:-1]:
            folder_id = child(folder_id, part)["id"]
        entry = child(folder_id, parts[-1])
        if entry.get("capabilities", {}).get("canDownload") is False:
            raise PermissionError(f"Download is disabled for {relative}")
        destination = processed / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        temporary = destination.with_name(destination.name + ".download")
        try:
            request = drive_api.files().get_media(fileId=entry["id"], supportsAllDrives=True)
            with temporary.open("wb") as stream:
                downloader = MediaIoBaseDownload(stream, request)
                done = False
                while not done:
                    _, done = downloader.next_chunk(num_retries=3)
            temporary.replace(destination)
        finally:
            temporary.unlink(missing_ok=True)
        print("Downloaded:", relative, flush=True)
except HttpError as error:
    raise RuntimeError(
        "Google Drive API access failed. Authenticate with the account that can open the "
        "shared folder, and check the API error below. No shortcut path is required. "
        + str(error)) from error

manifest = json.loads((processed / "manifest.json").read_text())
for name in required:
    actual = hashlib.sha256((processed / name).read_bytes()).hexdigest()
    if actual != manifest["processed_files"][name]["sha256"]:
        raise ValueError(f"Checksum mismatch: {name}. Restore the frozen team file.")
print("PASS: all nine consumed artifact checksums match the manifest.")


## 5. Build the configuration and preview all eight combinations
Re-run this cell after editing settings. A fresh output name is generated for every training execution, so old runs are preserved. The original team configuration is not overwritten.

In [ ]:
config = {
    "schema_version": 1,
    "data": {"processed_dir": str(processed),
             "expected_dataset": "Amazon Reviews 2023 - Beauty_and_Personal_Care",
             "expected_subset": "Skin Care", "max_seq_len": 50},
    "training": {"batch_size": BATCH_SIZE, "epochs": EPOCHS,
                 "num_layers": NUM_LAYERS, "num_heads": NUM_HEADS,
                 "weight_decay": WEIGHT_DECAY, "early_stopping_patience": PATIENCE,
                 "seeds": [42, 43, 44], "optimizer": OPTIMIZER, "scheduler": "none"},
    "search": {"hidden_size": [64, 128], "dropout": [0.2, 0.5],
               "learning_rate": [0.001, 0.0005]},
    "evaluation": {"selection_split": "val", "selection_metric": "NDCG@10", "sampled": False},
    "output_dir": str(PROJECT / "results")
}
config_path = PROJECT / "configs/colab_run.json"
config_path.write_text(json.dumps(config, indent=2) + "\n")
preview = subprocess.check_output([sys.executable, "-B", "-m", "src.training.grid_search",
    "--config", str(config_path), "--dry-run"], cwd=PROJECT, text=True)
for i, run in enumerate(json.loads(preview)):
    print(i, run["model_hyperparameters"])


## Verify initial loss before training
Checks both hidden sizes against chance loss on 128 real users and verifies the frozen data. This does not evaluate test data.

In [ ]:
from uuid import uuid4
preflight_path = PROJECT / "results" / ("preflight_" + uuid4().hex[:8] + ".json")
subprocess.run([sys.executable, "-B", "-m", "src.training.preflight",
               "--config", str(config_path), "--output", str(preflight_path)],
              cwd=PROJECT, check=True)

print(preflight_path.read_text())  # show the measured initial losses


## 6. Run the full eight-configuration search

This uses all training and validation users, seed 42, Adam, and no scheduler.
It saves results directly to Drive. Progress appears per configuration; epoch
losses and validation scores are in run_XX.log inside the output folder.
The first configuration in grid order wins an exact tie. Test data is not scored.


In [ ]:
subprocess.run([sys.executable, "-B", "-m", "src.training.environment_check",
    "--spec", str(PROJECT / "configs/colab_environment.json"),
    "--output", str(PROJECT / "environment_preflight.json")], cwd=PROJECT, check=True)
if RUN_MODE != "full" or TRAIN_USERS is not None or VALIDATION_USERS is not None:
    raise ValueError("Issue 3.1 requires full mode and no user limits.")
if not METRICS_4_4_COMMITTED:
    raise RuntimeError("Team prerequisite: confirm Workbook section 4.4 is committed, then set METRICS_4_4_COMMITTED=True and rerun settings.")
expected_training = {"batch_size": 128, "epochs": 200, "num_layers": 2, "num_heads": 1,
    "weight_decay": 0.0, "early_stopping_patience": 10, "seeds": [42,43,44],
    "optimizer": "Adam", "scheduler": "none"}
if config["training"] != expected_training or TUNING_SEED != 42 or EVAL_BATCH_SIZE != 128:
    raise ValueError("Restore the team's fixed settings and rerun the configuration cell.")
if DISCARD_RESUME_CHECKPOINTS:
    raise ValueError("Keep last.pt resume checkpoints for full training.")

from datetime import datetime, timezone
from uuid import uuid4

cuda_available = subprocess.check_output([sys.executable, "-c",
    "import torch; print(int(torch.cuda.is_available()))"], text=True).strip() == "1"
selected_device = ("cuda" if cuda_available else "cpu") if DEVICE == "auto" else DEVICE
if selected_device == "cuda" and not cuda_available:
    raise RuntimeError("CUDA is unavailable. Choose a GPU runtime or use DEVICE='cpu'.")
if RUN_MODE == "full" and selected_device != "cuda":
    raise RuntimeError("Use a GPU runtime for full tuning; keep CPU for smoke checks.")
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
run_root = DRIVE_RESULTS_DIR if RUN_MODE == "full" else PROJECT / "results"
run_root.mkdir(parents=True, exist_ok=True)
RUN_DIR = run_root / ("grid_" + RUN_MODE + "_" + run_id)
command = [sys.executable, "-B", "-m", "src.training.grid_search",
    "--config", str(config_path), "--seed", str(TUNING_SEED),
    "--optimizer", OPTIMIZER, "--device", selected_device,
    "--eval-batch-size", str(EVAL_BATCH_SIZE), "--output-dir", str(RUN_DIR)]
for flag, value in [("--limit-train-users", TRAIN_USERS), ("--limit-val-users", VALIDATION_USERS)]:
    if value is not None:
        command += [flag, str(value)]
if DISCARD_RESUME_CHECKPOINTS:
    command.append("--discard-last-checkpoint")
print("Device:", selected_device, "| Results:", RUN_DIR, flush=True)
completed = subprocess.run(command, cwd=PROJECT)
if RUN_DIR.exists():
    shutil.copy2(preflight_path, RUN_DIR / "preflight.json")
    shutil.copy2(PROJECT / "environment_preflight.json", RUN_DIR / "environment_preflight.json")
if completed.returncode:
    for log in sorted(RUN_DIR.glob("run_*.log"))[-1:]:
        print(log.name, log.read_text()[-6000:])
    raise RuntimeError("Search failed; no valid winner. Inspect logs below and save them if needed.")


## 7. Inspect the full-grid scores and selected configuration

Require eight PASS rows. A full run with all-zero scores needs investigation before reporting performance.


In [ ]:
import csv
from IPython.display import display, JSON

if (RUN_DIR / "grid_results.csv").exists():
    with (RUN_DIR / "grid_results.csv").open() as stream:
        for row in csv.DictReader(stream):
            print(row)
if (RUN_DIR / "selection.json").exists():
    display(JSON(json.loads((RUN_DIR / "selection.json").read_text())))
    display(JSON(json.loads((RUN_DIR / "best_config.json").read_text())))
else:
    print("No completed selection; inspect failure.json and run logs.")


## Verify the selected checkpoint
Checks its checksum, configuration, epoch, score and accepted evaluation policy; restores the model without evaluating test data. Run after a successful grid.

In [ ]:
subprocess.run([sys.executable, "-B", "-m", "src.training.checkpoints",
                "--run-dir", str(RUN_DIR), "--device", selected_device],
               cwd=PROJECT, check=True)


## 8. Save the run to Drive
Run this after completion or failure. Runtime files are temporary. This copies logs, configs, metrics and checkpoints to a new Drive folder and refuses to overwrite an existing folder. If copying fails partway, use a new destination name for retry. Keep checkpoint binaries out of GitHub.

In [ ]:
destination = DRIVE_RESULTS_DIR / RUN_DIR.name
DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
if destination.resolve() == RUN_DIR.resolve():
    print("Full run already saved directly to Drive:", RUN_DIR)
else:
    shutil.copytree(RUN_DIR, destination)
    print("Saved to Drive:", destination)


## 9. Prepare repeat training using the existing Part E runner

Run these source cells too. They copy the team's evaluation helpers unchanged.
Only the runner's **train** command is used; it does not evaluate test data.
Seed 42 is reused from the winning grid run. Seeds 43 and 44 use exactly the same
configuration and data. Coordinate with Sneha to avoid launching duplicate repeats.


### src/evaluation/common.py

In [ ]:
%%writefile /content/issue2_grid_project/src/evaluation/common.py
"""Shared helpers for Parts E and F that do not need PyTorch.

`load_evaluator` repeats the manifest check of src/training/integration_check.py
because that module imports torch at import time; the popularity and chance
references in Part F must run in an environment without it.
"""
import hashlib
import importlib.util
import json
from pathlib import Path

import numpy as np
import pandas as pd

from src.training.config import GRID, expand_grid, validate_config

# Same nine files as integration_check.REQUIRED, plus what Parts E and F read.
EVALUATOR_FILES = ['dataset_info.json', 'rec_eval.py', 'sequences.parquet',
                   'eval/eval_protocol.json', 'eval/neg_val.npy', 'eval/neg_test.npy',
                   'eval/eval_cases.parquet', 'groups/item_groups.parquet', 'groups/user_groups.parquet']
CATEGORY_FILE = 'item_text.parquet'
COLD_FILES = ['cold_start/cold_items.npy', 'cold_start/cold_sequences.parquet',
              'cold_start/cold_start_info.json']


def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def verify_files(processed_dir, names):
    """Check each file against the frozen manifest and return its SHA-256."""
    path = Path(processed_dir)
    manifest = json.loads((path / 'manifest.json').read_text())
    checked = {}
    for name in names:
        actual = digest(path / name)
        if actual != manifest['processed_files'][name]['sha256']:
            raise ValueError(f'Manifest checksum mismatch: {name}')
        checked[name] = actual
    return checked


def load_evaluator(processed_dir, extra_files=()):
    """Verify, then import the frozen rec_eval.py and load the dataset with it."""
    hashes = verify_files(processed_dir, [*EVALUATOR_FILES, *extra_files])
    spec = importlib.util.spec_from_file_location('shared_rec_eval', Path(processed_dir) / 'rec_eval.py')
    evaluator = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(evaluator)
    return evaluator, evaluator.load_dataset(processed_dir), hashes


def item_categories(processed_dir, data):
    """Category per item_idx (index 0 = padding), built exactly as notebook 07 does."""
    items = pd.read_parquet(Path(processed_dir) / CATEGORY_FILE).sort_values('item_idx')
    if not np.array_equal(items.item_idx.to_numpy(), np.arange(1, data['n_items'] + 1)):
        raise ValueError('item_text.parquet does not cover item_idx 1..n_items exactly')
    column = 'cat_l2' if data['info']['subset']['kind'] == 'category' else 'cat_l1'
    return np.array(['padding'] + items[column].fillna('unknown').tolist(), dtype=object)


def json_ready(value):
    """Convert NumPy scalars/arrays so json.dumps(allow_nan=False) accepts them."""
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(v) for v in value]
    if isinstance(value, np.ndarray):
        return json_ready(value.tolist())
    if isinstance(value, np.generic):
        return value.item()
    return value


def load_run(run_dir):
    """Settings, metadata and summary of one completed Part C run directory."""
    run_dir = Path(run_dir)
    summary = json.loads((run_dir / 'summary.json').read_text())
    if summary.get('status') != 'PASS':
        raise ValueError(f'{run_dir} did not complete')
    run_config = json.loads((run_dir / 'run_config.json').read_text())
    return run_config['training'], run_config['metadata'], summary


def expected_configuration(config_path, grid_index=None):
    """The team-confirmed settings every final run must have used.

    Mirrors src.training.train.resolve_config, which cannot be imported without PyTorch:
    a grid config needs --grid-index, an expanded config (one dict from expand_grid) does not.
    Empty (null) settings are rejected.
    """
    config = json.loads(Path(config_path).read_text())
    if 'search' in config:
        validate_config(config, require_ready=True)
        runs = expand_grid(config)
        if grid_index is None or not 0 <= grid_index < len(runs):
            raise ValueError(f'Select the configuration with --grid-index 0..{len(runs) - 1}')
        return runs[grid_index]
    if grid_index is not None:
        raise ValueError('An expanded configuration does not take --grid-index')
    validate_config({**config, 'search': GRID}, require_ready=True)
    parameters = config.get('model_hyperparameters', {})
    if set(parameters) != set(GRID) or any(parameters[k] not in GRID[k] for k in GRID):
        raise ValueError('Invalid expanded model_hyperparameters')
    return config


def configuration_mismatches(training, metadata, expected):
    """Every difference between one run's recorded settings and the expected configuration."""
    settings, hyper = expected['training'], expected['model_hyperparameters']
    pairs = {'model_hyperparameters': (metadata.get('model_hyperparameters'), hyper),
             'learning_rate': (training.get('learning_rate'), hyper['learning_rate']),
             'batch_size': (metadata.get('batch_size'), settings['batch_size']),
             'epochs': (training.get('epochs'), settings['epochs']),
             'num_layers': (metadata.get('model_layers'), settings['num_layers']),
             'num_heads': (metadata.get('model_heads'), settings['num_heads']),
             'weight_decay': (training.get('weight_decay'), settings['weight_decay']),
             'early_stopping_patience': (training.get('patience'),
                                         settings['early_stopping_patience']),
             'max_seq_len': (metadata.get('max_seq_len'), expected['data']['max_seq_len'])}
    # Older development configs did not declare these settings. Enforce them
    # whenever the team configuration does, including explicit scheduler absence.
    if 'optimizer' in settings:
        pairs['optimizer'] = (training.get('optimizer'), settings['optimizer'])
    if settings.get('scheduler') == 'none':
        for name in ('lr_step_size', 'lr_gamma'):
            pairs[name] = (training.get(name, '<missing>'), None)
    found = [f'{name}: run {run!r}, config {wanted!r}'
             for name, (run, wanted) in pairs.items() if run != wanted]
    if training.get('seed') not in settings['seeds']:
        found.append(f"seed {training.get('seed')!r} is not in the config seeds {settings['seeds']}")
    return found


def partial_reasons(metadata, n_train_users, n_validation_users):
    """Why a run counts as partial: trained or validated on fewer users than the full split."""
    reasons = []
    if metadata['train_users'] != n_train_users:
        reasons.append(f"trained on {metadata['train_users']:,} of {n_train_users:,} users")
    if metadata['validation_users'] != n_validation_users:
        reasons.append(f"validated on {metadata['validation_users']:,} of {n_validation_users:,} users")
    return reasons


def write_new(path, text):
    """Write a file that must not exist yet (results are never silently overwritten)."""
    with Path(path).open('x') as stream:
        stream.write(text)


### src/evaluation/seed_stats.py

In [ ]:
%%writefile /content/issue2_grid_project/src/evaluation/seed_stats.py
"""Part E: aggregate per-seed metrics into mean and sample standard deviation.

Pure NumPy/pandas, no model code. Used for the main Model 0 seeds and the cold-start runs.
"""
import math

import numpy as np
import pandas as pd

GROUP_KEYS = ['grouping', 'group']
TABLE_METRICS = ['Recall@10', 'NDCG@10', 'Recall@20', 'NDCG@20', 'MRR', 'Coverage@10']


def summarize(values):
    """Mean and sample std (ddof=1). The std is None for a single run, never a fake 0."""
    values = np.asarray(values, dtype=float)
    if not len(values) or not np.isfinite(values).all():
        raise ValueError('Need at least one finite value per metric')
    return {'mean': float(values.mean()),
            'std': float(values.std(ddof=1)) if len(values) > 1 else None,
            'min': float(values.min()), 'max': float(values.max()), 'n': int(len(values))}


def aggregate_metrics(per_seed):
    """per_seed: {seed: {metric: value}}. Every seed must report the same metrics."""
    if len(per_seed) < 1:
        raise ValueError('No runs to aggregate')
    names = [set(m) for m in per_seed.values()]
    if any(n != names[0] for n in names):
        raise ValueError('Seeds report different metric sets')
    seeds = sorted(per_seed)
    return {metric: {**summarize([per_seed[s][metric] for s in seeds]),
                     'values': {str(s): float(per_seed[s][metric]) for s in seeds}}
            for metric in sorted(names[0])}


def aggregate_groups(per_seed):
    """per_seed: {seed: DataFrame from rec_eval.group_breakdown}. Case counts must agree."""
    frames = []
    for seed, frame in sorted(per_seed.items()):
        frames.append(pd.DataFrame(frame).assign(seed=seed))
    rows = pd.concat(frames, ignore_index=True)
    counts = rows.groupby(GROUP_KEYS).n_cases.nunique()
    if (counts != 1).any():
        raise ValueError('Group case counts differ between seeds; runs used different users')
    metrics = [c for c in rows.columns if c not in (*GROUP_KEYS, 'n_cases', 'seed')]
    out = []
    for (grouping, group), d in rows.groupby(GROUP_KEYS, sort=False):
        row = {'grouping': grouping, 'group': group, 'n_cases': int(d.n_cases.iloc[0]),
               'n_seeds': len(d)}
        for metric in metrics:
            s = summarize(d[metric])
            row[f'{metric}_mean'], row[f'{metric}_std'] = s['mean'], s['std']
        out.append(row)
    return pd.DataFrame(out)


def mean_std(summary, digits=4):
    """'0.0412 ± 0.0011', or just the mean when only one seed exists or std is undefined."""
    if summary['std'] is None:
        return f"{summary['mean']:.{digits}f}"
    return f"{summary['mean']:.{digits}f} ± {summary['std']:.{digits}f}"


def markdown_table(rows, metrics=TABLE_METRICS, label='Model', digits=None):
    """rows: [(name, aggregate_metrics(...) result)]. Missing metrics print as a blank."""
    digits = digits or {}
    header = f"| {label} | " + ' | '.join(metrics) + ' |'
    lines = [header, '|' + '---|' * (len(metrics) + 1)]
    for name, aggregate in rows:
        cells = [mean_std(aggregate[m], digits.get(m, 4)) if m in aggregate else '' for m in metrics]
        lines.append(f'| {name} | ' + ' | '.join(cells) + ' |')
    return '\n'.join(lines) + '\n'


def single_run(metrics):
    """Wrap deterministic baselines (popularity, chance) so they print in the same table."""
    return {k: {'mean': float(v), 'std': None, 'min': float(v), 'max': float(v), 'n': 1}
            for k, v in metrics.items() if isinstance(v, (int, float)) and math.isfinite(v)}


### src/evaluation/multi_seed.py

In [ ]:
%%writefile /content/issue2_grid_project/src/evaluation/multi_seed.py
"""Part E: repeat ONE selected configuration over several seeds, then score test once per seed.

`train` runs Part C (src.training.train) unchanged, one subprocess per seed.
`evaluate` first checks every run matches the team configuration and used all
users. It then reloads each best.pt, checks it reproduces its own validation
score, and calls the frozen rec_eval.py on test: full catalog (main protocol), 100
sampled negatives (secondary), user/item group breakdown and category diversity.
Each run's test result is written once next to its checkpoint; rerunning
`evaluate` reuses it instead of scoring the test split again.
"""
import argparse
from datetime import datetime, timezone
import json
from pathlib import Path
import platform
import subprocess
import sys

import numpy as np
import pandas as pd
import torch

from src.evaluation.common import (CATEGORY_FILE, configuration_mismatches, digest,
                                   expected_configuration, item_categories, json_ready, load_run,
                                   partial_reasons, verify_files, write_new)
from src.evaluation.seed_stats import aggregate_groups, aggregate_metrics, markdown_table, mean_std
from src.models.sasrec import SASRec
from src.training.config import load_config
from src.training.integration_check import load_verified_data
from src.training.trainer import load_best

ROOT = Path(__file__).resolve().parents[2]
# Reloaded best.pt must reproduce the validation NDCG@10 recorded during training.
VALIDATION_TOLERANCE = 1e-4
GROUP_METRICS = ['Recall@10', 'NDCG@10', 'MRR']


def train_seeds(runs_dir, seeds, forwarded, allow_partial=False):
    """Call Part C once per seed. Completed seeds are skipped, failed ones are reported."""
    if any(flag in forwarded for flag in ('--seed', '--output-dir', '--resume')):
        raise ValueError('--seed, --output-dir and --resume are set per seed by this runner')
    limits = [flag for flag in forwarded if flag.startswith('--limit-')]
    if limits and not allow_partial:
        raise ValueError(f'{", ".join(limits)} trains or validates on a user subset; '
                         'add --allow-partial-runs for development runs')
    runs_dir = Path(runs_dir)
    for seed in seeds:
        output = runs_dir / f'seed_{seed}'
        if (output / 'summary.json').exists():
            print(f'seed {seed}: already trained, skipping {output}')
            continue
        if output.exists():
            raise FileExistsError(f'{output} exists without summary.json. Resume it with '
                                  'src.training.train --resume into a new directory, then pass '
                                  'that directory to evaluate.')
        command = [sys.executable, '-m', 'src.training.train', *forwarded,
                   '--seed', str(seed), '--output-dir', str(output)]
        print('Running:', ' '.join(command), flush=True)
        subprocess.run(command, check=True, cwd=ROOT)


def build_model(metadata, n_items):
    hyper = metadata['model_hyperparameters']
    return SASRec(n_items, max_len=metadata['max_seq_len'], hidden_size=hyper['hidden_size'],
                  dropout=hyper['dropout'], num_heads=metadata['model_heads'],
                  num_layers=metadata['model_layers'])


def scorer(model, n_items, device):
    """score_fn in the rec_eval contract: (inputs, users) -> (batch, n_items + 1) NumPy."""
    @torch.no_grad()
    def score_fn(inputs, users):
        del users
        model.eval()
        scores = model.next_item_scores(torch.as_tensor(inputs, dtype=torch.long, device=device))
        if scores.shape != (len(inputs), n_items + 1) or not torch.isfinite(scores).all():
            raise ValueError('Invalid/nonfinite score matrix')
        return scores.float().cpu().numpy()
    return score_fn


def comparable(training, metadata):
    """Everything that must match across seeds: all settings except seed/device, all
    provenance except the Git commit (a README-only commit between seeds is harmless;
    source file hashes are still compared)."""
    return ({k: v for k, v in training.items() if k not in ('seed', 'device')},
            {k: v for k, v in metadata.items() if k != 'commit'})


def check_runs(run_dirs, data, hashes, manifest_sha256, allow_partial, expected=None):
    """Refuse to aggregate runs that are not the same experiment on the full data.

    expected: the team configuration (expected_configuration); every run must match it.
    """
    runs = [load_run(d) for d in run_dirs]
    seeds = [training['seed'] for training, _, _ in runs]
    if len(set(seeds)) != len(seeds):
        raise ValueError(f'Duplicate seeds: {seeds}')
    reference = comparable(*runs[0][:2])
    for run_dir, (training, metadata, _) in zip(run_dirs, runs):
        if comparable(training, metadata) != reference:
            raise ValueError(f'{run_dir} differs from {run_dirs[0]} in more than seed/device')
        if metadata['manifest_sha256'] != manifest_sha256 or metadata['verified_files'] != hashes:
            raise ValueError(f'{run_dir} was trained on different frozen data')
        partial = partial_reasons(metadata, data['n_users'], data['n_users'])
        if partial and not allow_partial:
            raise ValueError(f'{run_dir} is partial ({"; ".join(partial)}); '
                             'only --allow-partial-runs accepts it')
        if expected is not None:
            mismatches = configuration_mismatches(training, metadata, expected)
            if mismatches:
                raise ValueError(f'{run_dir} does not match the team configuration: '
                                 + '; '.join(mismatches))
    return runs


def evaluate_run(run_dir, run, evaluator, data, item_cat, device, batch_size):
    run_dir = Path(run_dir)
    training, metadata, summary = run
    record_path, ranks_path = run_dir / 'test_evaluation.json', run_dir / 'test_ranks.npz'
    checkpoint_sha256 = digest(run_dir / 'best.pt')
    if record_path.exists():
        record = json.loads(record_path.read_text())
        if record['checkpoint_sha256'] != checkpoint_sha256:
            raise ValueError(f'{run_dir}/best.pt changed after its test evaluation')
        print(f'seed {training["seed"]}: reusing existing test evaluation')
        return record
    if ranks_path.exists():
        raise FileExistsError(f'{ranks_path} exists without test_evaluation.json; inspect it first')
    model = build_model(metadata, data['n_items'])
    checkpoint = load_best(model, run_dir / 'best.pt', device=device)
    if checkpoint['metadata'] != metadata or checkpoint['epoch'] != summary['best_epoch']:
        raise ValueError(f'{run_dir}/best.pt does not belong to this run')
    score_fn = scorer(model, data['n_items'], device)
    # Same validation users and batch size as during training, then compare scores.
    users = data['sequences'].user_idx.iloc[:metadata['validation_users']].to_numpy()
    val = evaluator.evaluate(score_fn, data, split='val', sampled=False, users=users,
                             batch_size=metadata['eval_batch_size'], return_topk=False)
    gap = abs(val['metrics']['NDCG@10'] - summary['best_validation_NDCG@10'])
    if gap > VALIDATION_TOLERANCE:
        raise ValueError(f'{run_dir}: reloaded validation NDCG@10 differs by {gap:.2e}')
    test = evaluator.evaluate(score_fn, data, split='test', sampled=False, batch_size=batch_size)
    sampled = evaluator.evaluate(score_fn, data, split='test', sampled=True, batch_size=batch_size)
    full_metrics = {**test['metrics'],
                    'ILD@10': evaluator.category_diversity(test['topk'][:, :10], item_cat)}
    groups = evaluator.group_breakdown(test, data)
    np.savez_compressed(ranks_path, user_idx=test['user_idx'], ranks=test['ranks'], topk=test['topk'])
    record = {'seed': training['seed'], 'run_dir': str(run_dir), 'checkpoint_sha256': checkpoint_sha256,
              'best_epoch': summary['best_epoch'], 'purpose': metadata['purpose'],
              'legacy_tie_policy_unresolved': metadata.get('legacy_tie_policy_unresolved'),
              'tie_policy': metadata.get('tie_policy'),
              'validation_check': {'saved_NDCG@10': summary['best_validation_NDCG@10'],
                                   'recomputed_NDCG@10': val['metrics']['NDCG@10'], 'gap': gap},
              'n_test_cases': test['n_cases'], 'test_full': full_metrics,
              'test_sampled': sampled['metrics'], 'groups': groups.to_dict('records'),
              'ranks_file': ranks_path.name, 'device': device,
              'evaluated_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
              'versions': {'python': platform.python_version(), 'torch': str(torch.__version__),
                           'numpy': np.__version__, 'pandas': pd.__version__}}
    write_new(record_path, json.dumps(json_ready(record), indent=2, allow_nan=False) + '\n')
    print(f'seed {training["seed"]}: test NDCG@10 = {full_metrics["NDCG@10"]:.4f}', flush=True)
    return record


def write_aggregate(records, output_dir, label):
    full = aggregate_metrics({r['seed']: r['test_full'] for r in records})
    sampled = aggregate_metrics({r['seed']: r['test_sampled'] for r in records})
    groups = aggregate_groups({r['seed']: pd.DataFrame(r['groups']) for r in records})
    seeds = sorted(r['seed'] for r in records)
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=False)
    write_new(output / 'aggregate.json', json.dumps(json_ready({
        'label': label, 'seeds': seeds, 'n_test_cases': records[0]['n_test_cases'],
        'std': 'sample standard deviation (ddof=1) across seeds',
        'purpose': sorted({r['purpose'] for r in records}),
        'runs': {str(r['seed']): r['run_dir'] for r in records},
        'test_full': full, 'test_sampled': sampled}), indent=2, allow_nan=False) + '\n')
    groups.to_csv(output / 'group_breakdown.csv', index=False)
    lines = [f'# {label}: test results over seeds {", ".join(map(str, seeds))}', '',
             f'Mean ± sample standard deviation over {len(seeds)} seeds, '
             f'{records[0]["n_test_cases"]:,} test users. Selection used validation NDCG@10 only.', '',
             markdown_table([(f'{label}, full catalog', full),
                             (f'{label}, 100 sampled negatives', sampled)],
                            metrics=['Recall@10', 'NDCG@10', 'Recall@20', 'NDCG@20', 'MRR']),
             markdown_table([(f'{label}, full catalog', full)],
                            metrics=['Coverage@10', 'Coverage@20', 'ILD@10']),
             '| Grouping | Group | Cases | ' + ' | '.join(GROUP_METRICS) + ' |',
             '|---|---|---|' + '---|' * len(GROUP_METRICS)]
    for row in groups.to_dict('records'):
        cells = [mean_std({'mean': row[f'{m}_mean'], 'std': row[f'{m}_std']}) for m in GROUP_METRICS]
        lines.append(f"| {row['grouping']} | {row['group']} | {row['n_cases']:,} | "
                     + ' | '.join(cells) + ' |')
    write_new(output / 'evaluation_table.md', '\n'.join(lines) + '\n')
    print((output / 'evaluation_table.md').read_text())


def evaluate(args):
    config = load_config(args.config)
    processed = Path(config['data']['processed_dir'])
    evaluator, data, hashes = load_verified_data(processed, config)
    verify_files(processed, [CATEGORY_FILE])
    if Path(args.output_dir).exists():
        raise FileExistsError('Output directory exists; choose a new one')
    expected = expected_configuration(args.config, args.grid_index)
    runs = check_runs(args.run_dirs, data, hashes, digest(processed / 'manifest.json'),
                      args.allow_partial_runs, expected)
    item_cat = item_categories(processed, data)
    records = [evaluate_run(d, run, evaluator, data, item_cat, args.device, args.eval_batch_size)
               for d, run in zip(args.run_dirs, runs)]
    write_aggregate(records, args.output_dir, args.label)


def main():
    parser = argparse.ArgumentParser(description=__doc__,
                                     formatter_class=argparse.RawDescriptionHelpFormatter)
    commands = parser.add_subparsers(dest='command', required=True)
    # allow_abbrev=False: a forwarded --seed must not be read as an abbreviation of --seeds.
    train = commands.add_parser('train', allow_abbrev=False,
                                help='Run src.training.train once per seed; other arguments '
                                'are forwarded to it unchanged')
    train.add_argument('--runs-dir', required=True, help='Each seed writes RUNS_DIR/seed_<seed>')
    train.add_argument('--seeds', type=int, nargs='+', required=True)
    train.add_argument('--allow-partial-runs', action='store_true',
                       help='Allow forwarded --limit-* flags (development runs only)')
    score = commands.add_parser('evaluate', allow_abbrev=False,
                                help='Score each run on test once, then aggregate')
    score.add_argument('--run-dirs', nargs='+', required=True)
    score.add_argument('--config', default='configs/config.json',
                       help='Team configuration every run must match')
    score.add_argument('--grid-index', type=int, help='Selected grid configuration (grid configs)')
    score.add_argument('--output-dir', required=True)
    score.add_argument('--label', default='Model 0 (SASRec)')
    score.add_argument('--device', choices=['cpu', 'cuda', 'mps'], default='cpu')
    score.add_argument('--eval-batch-size', type=int, default=256)
    score.add_argument('--allow-partial-runs', action='store_true',
                       help='Accept development runs trained or validated on a user subset')
    args, forwarded = parser.parse_known_args()
    try:
        if args.command == 'train':
            train_seeds(args.runs_dir, args.seeds, forwarded, args.allow_partial_runs)
        else:
            if forwarded:
                parser.error(f'Unknown arguments: {forwarded}')
            evaluate(args)
    except (ValueError, FileNotFoundError, FileExistsError) as error:
        parser.exit(2, f'Part E stopped: {error}\n')
    except subprocess.CalledProcessError as error:
        parser.exit(error.returncode, f'Part E stopped: training failed ({error})\n')


if __name__ == '__main__':
    main()


In [ ]:
subprocess.run([sys.executable, "-B", "-m", "src.training.environment_check",
    "--spec", str(PROJECT / "configs/colab_environment.json"),
    "--output", str(PROJECT / "environment_preflight.json")], cwd=PROJECT, check=True)
shutil.copy2(PROJECT / "environment_preflight.json", RUN_DIR / "environment_preflight_repeats.json")
selection = json.loads((RUN_DIR / "selection.json").read_text())
if (RUN_MODE != "full" or selection.get("user_limited") is not False
        or selection.get("completed_runs") != 8 or selection.get("status") != "PASS"
        or selection.get("seed") != 42 or selection.get("test_evaluated") is not False):
    raise ValueError("Repeats require a successful full eight-run seed-42 grid.")
if selection["best_validation_NDCG@10"] <= 0:
    raise ValueError("Full-grid winner scored zero. Investigate the logs before repeating it.")
repeat_config = json.loads((RUN_DIR / "best_config.json").read_text())
repeat_config["training"]["seeds"] = [42, 43, 44]
repeat_path = RUN_DIR / "winner_repeat_config.json"
if repeat_path.exists():
    if json.loads(repeat_path.read_text()) != repeat_config:
        raise ValueError("Existing repeat config differs; preserve it and investigate.")
else:
    repeat_path.write_text(json.dumps(repeat_config, indent=2) + "\n")
REPEAT_DIR = RUN_DIR / "winner_repeats"
print("Reusing seed 42:", RUN_DIR / Path(selection["source_checkpoint"]).parent)
print("Training seeds 43 and 44 into:", REPEAT_DIR)
subprocess.run([sys.executable, "-u", "-B", "-m", "src.evaluation.multi_seed", "train",
    "--runs-dir", str(REPEAT_DIR), "--seeds", "43", "44",
    "--config", str(repeat_path), "--optimizer", "Adam", "--device", selected_device,
    "--eval-batch-size", str(EVAL_BATCH_SIZE)], cwd=PROJECT, check=True)


## 10. Verify all three saved models and save the validation summary

This checks identical configuration, source versions and frozen data, all-user
coverage, and checkpoint validation-score reproduction. It saves per-seed scores,
mean and sample standard deviation, checkpoint hashes, and all run paths.
The validation mean includes seed 42, which selected the winner, so it is not an
unbiased final test estimate. Send these run folders to Sneha for Part E test scoring.


In [ ]:
verification_script = "import json, math, statistics\nfrom pathlib import Path\nimport torch\nfrom src.evaluation.common import digest, expected_configuration, json_ready\nfrom src.evaluation.multi_seed import check_runs, build_model, scorer, VALIDATION_TOLERANCE\nfrom src.training.integration_check import load_verified_data\nfrom src.training.trainer import load_best\nfrom src.training.train import TIE_POLICY\n\nroot = Path(__import__('sys').argv[1])\nconfig_path = root / 'winner_repeat_config.json'\nconfig = expected_configuration(config_path)\nselection = json.loads((root / 'selection.json').read_text())\nrun_dirs = [root / Path(selection['source_checkpoint']).parent,\n            root / 'winner_repeats/seed_43', root / 'winner_repeats/seed_44']\nevaluator, data, hashes = load_verified_data(config['data']['processed_dir'], config)\nruns = check_runs(run_dirs, data, hashes,\n    digest(Path(config['data']['processed_dir']) / 'manifest.json'), False, config)\nif [r[0]['seed'] for r in runs] != [42,43,44]:\n    raise ValueError('Require seeds 42,43,44 in order')\nrows = []\nfor directory, (training, metadata, summary) in zip(run_dirs, runs):\n    if (summary['test_evaluated'] is not False or metadata.get('tie_policy') != TIE_POLICY\n        or training['optimizer'] != 'Adam' or training['lr_step_size'] is not None\n        or training['lr_gamma'] is not None):\n        raise ValueError('Unexpected evaluation or optimizer settings')\n    device = 'cuda' if torch.cuda.is_available() else 'cpu'\n    model = build_model(metadata, data['n_items'])\n    checkpoint = load_best(model, directory / 'best.pt', device=device)\n    if (checkpoint['metadata'] != metadata or checkpoint['epoch'] != summary['best_epoch']\n        or checkpoint['validation_NDCG@10'] != summary['best_validation_NDCG@10']):\n        raise ValueError('Checkpoint does not match run summary')\n    val = evaluator.evaluate(scorer(model, data['n_items'], device), data,\n        split='val', sampled=False, users=data['sequences'].user_idx.to_numpy(),\n        batch_size=metadata['eval_batch_size'], return_topk=False)\n    score = float(val['metrics']['NDCG@10'])\n    if not math.isfinite(score) or abs(score-summary['best_validation_NDCG@10']) > VALIDATION_TOLERANCE:\n        raise ValueError('Reloaded validation score does not match')\n    rows.append({'seed':training['seed'], 'run_dir':str(directory),\n        'best_epoch':summary['best_epoch'], 'saved_validation_NDCG@10':summary['best_validation_NDCG@10'],\n        'recomputed_validation_NDCG@10':score, 'checkpoint_sha256':digest(directory/'best.pt')})\n    print(rows[-1], flush=True)\n    del model, checkpoint\n    if torch.cuda.is_available(): torch.cuda.empty_cache()\nscores = [r['recomputed_validation_NDCG@10'] for r in rows]\nreport = {'status':'PASS', 'scope':'full_grid_and_winner_repeats_validation_only',\n    'test_evaluated':False, 'model_hyperparameters':config['model_hyperparameters'],\n    'runs':rows, 'validation_NDCG@10_mean':statistics.mean(scores),\n    'validation_NDCG@10_sample_std':statistics.stdev(scores),\n    'note':'Seed 42 selected the winner; this validation aggregate is not a final test estimate.'}\n(root/'issue3_1_validation_summary.json').write_text(json.dumps(json_ready(report),indent=2,allow_nan=False)+'\\n')\nprint(json.dumps(report,indent=2))\n"
verification_path = PROJECT / "verify_winner_repeats.py"
verification_path.write_text(verification_script)
subprocess.run([sys.executable, "-u", "-B", str(verification_path), str(RUN_DIR)], cwd=PROJECT, check=True)

validation_report_path = RUN_DIR / "issue3_1_validation_summary.json"
print(validation_report_path.read_text())


## Completion evidence and handoff

Keep `grid_results.csv`, `selection.json`, `best_config.json`, `preflight.json`,
`winner_repeat_config.json`, `issue3_1_validation_summary.json`, and the complete
winning seed-42/43/44 run folders (including best.pt, run_config.json, history.csv,
summary.json and last.pt). All are saved under the printed Drive output folder.

Eight full grid runs plus the two additional winner seeds complete the training
portion of 3.1. Final test metrics still require the team's Part E evaluation,
including its category data file. Do not report validation scores as test scores.
Keep checkpoints and datasets in Drive; use GitHub for code and small reports.

For Part E, share the complete winning seed-42 folder and winner_repeats/seed_43
and seed_44. Confirm the recipient can open the folders; do not change data or
checkpoint permissions to public. Include environment_preflight.json and
 environment_preflight_repeats.json for new runs. The historical November 7 run
predates these checks; its original runtime metadata must remain unchanged.
